# 01 — Data acquisition

Download comparable national health-financing indicators for India using the World Bank Indicators API. The health-financing source is WHO's Global Health Expenditure Database. These are **national time series, not state-level observations**.

In [ ]:
import requests
from datetime import date
from pathlib import Path
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/"README.md").exists() and (ROOT.parent/"README.md").exists(): ROOT=ROOT.parent
RAW=ROOT/"Dataset"/"Raw_Data"; OUT=ROOT/"Dataset"/"Processed_Data"; FIG=ROOT/"Dataset"/"Notebooks"/"Visualisations"
RAW.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True); FIG.mkdir(parents=True,exist_ok=True)
INDICATORS={"oop_share":"SH.XPD.OOPC.CH.ZS","oop_pc_usd":"SH.XPD.OOPC.PC.CD","health_gdp":"SH.XPD.CHEX.GD.ZS","gov_share":"SH.XPD.GHED.CH.ZS","health_pc_usd":"SH.XPD.CHEX.PC.CD"}
rows=[]
for name,code in INDICATORS.items():
    url=f"https://api.worldbank.org/v2/country/IND/indicator/{code}"
    response=requests.get(url,params={"format":"json","per_page":100},timeout=60); response.raise_for_status()
    payload=response.json()
    if not isinstance(payload,list) or len(payload)<2 or payload[1] is None: raise ValueError(f"No data for {code}: {payload}")
    for x in payload[1]: rows.append({"country":x["country"]["value"],"country_code":x["countryiso3code"],"year":int(x["date"]),"indicator":name,"indicator_code":code,"value":x["value"],"source":"World Bank API / WHO GHED"})
df=pd.DataFrame(rows).sort_values(["indicator","year"])
df.to_csv(RAW/"india_health_financing_world_bank.csv",index=False)
pd.DataFrame([{"indicator":k,"code":v,"retrieved_on":date.today().isoformat()} for k,v in INDICATORS.items()]).to_csv(RAW/"world_bank_indicator_metadata.csv",index=False)
display(df.tail(10))

## Sources and definitions
- World Bank API documentation: https://datahelpdesk.worldbank.org/knowledgebase/articles/898581
- OOP share definition and data: https://data.worldbank.org/indicator/SH.XPD.OOPC.CH.ZS
- WHO Global Health Expenditure Database: https://apps.who.int/nha/database/

Latest available year can differ by indicator. Missing values are not zero. The API may revise historical estimates.